# AOGC · MIRAGE rollout + judge (Colab GPU side)

This is the **GPU half** of the AOGC pipeline. It:
1. loads MIRAGE-Bench grounding snapshots (the loader),
2. runs a local 4-bit model to produce `<think>/<action>`,
3. uses an LLM-judge to label `is_error` (MIRAGE ships no label),
4. appends each scored trajectory to a **resumable** `.jsonl` on Google Drive.

Then either run the last cell here, or pull the `.jsonl` to your Mac and run
`aogc_uq` analysis (AOGC / FRANQ / metrics) — no GPU needed there.

**Set `Runtime → Change runtime type → GPU (T4)` first.** A disconnect loses at
most one trajectory; just re-run the rollout cell to resume.

In [ ]:
# 1. deps (torch is preinstalled on Colab). sentencepiece works here (unlike the PEP-668 Mac).
!pip -q install -U transformers accelerate bitsandbytes sentencepiece scikit-learn anthropic openai

In [ ]:
# 2. GPU check
import torch
print("CUDA:", torch.cuda.is_available(),
      "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU -> Runtime>Change runtime type>GPU"))

In [ ]:
# 3. mount Drive (for resumable output + model cache across sessions)
from google.colab import drive
drive.mount('/content/drive')

## Config
**No secrets? Both are optional:**
- **Clone without a GitHub token**: upload `AOGC-agent-uq.zip` to Drive at
  `MyDrive/AOGC-agent-uq.zip` (the clone cell auto-unzips it). Or make the repo public.
- **Judge without an API key**: keep `JUDGE_BACKEND = "local"` (reuses the rollout
  model on the same GPU). `GITHUB_TOKEN`/`ANTHROPIC_API_KEY` are only needed for the
  token-clone / API-judge paths.

In [ ]:
# 4. config
AOGC_REPO_URL = "https://github.com/Mowon0303/AOGC-agent-uq"   # PRIVATE -> needs GITHUB_TOKEN secret (see clone cell)
AOGC_LOCAL_DIR = "/content/drive/MyDrive/AOGC-agent-uq"  # fallback if you uploaded instead of pushing
MIRAGE_URL    = "https://github.com/sunblaze-ucb/mirage-bench"

MODEL        = "Qwen/Qwen2.5-3B-Instruct"          # T4-safe. 7B may OOM on long MIRAGE prompts; raise GPU/Pro for 7B
SETTINGS     = ["popup", "unexpected_transition", "misleading", "error_feedback"]  # grounding slice
ENVIRONMENTS = None        # None=all; or e.g. ["webarena"]
LIMIT        = 50          # snapshots to score this run; raise once it works

OUT_DIR      = "/content/drive/MyDrive/aogc_runs"
HF_CACHE     = "/content/drive/MyDrive/hf_cache"   # cache 7B weights across sessions

JUDGE_BACKEND = "local"                      # "local"(免API key,复用rollout模型当judge) | "anthropic" | "openai" | "stub"
JUDGE_MODEL   = "claude-haiku-4-5-20251001"  # cheap, strong enough for the judge
# "local" needs NO key. For "anthropic"/"openai" put the key in Colab Secrets (left sidebar key icon).

In [ ]:
# 5. clone repos + install aogc_uq
import os, sys, subprocess
os.makedirs(OUT_DIR, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE

def sh(*a): print("+", " ".join(a)); subprocess.run(a, check=True)

# MIRAGE-Bench data
if not os.path.isdir("/content/mirage-bench"):
    sh("git", "clone", "--depth", "1", MIRAGE_URL, "/content/mirage-bench")
os.environ["MIRAGE_ROOT"] = "/content/mirage-bench/dataset_all"

# AOGC repo
aogc_dir = None
if AOGC_REPO_URL:
    if not os.path.isdir("/content/aogc"):
        url = AOGC_REPO_URL
        try:  # private repo: prepend a token from Colab Secrets if available
            from google.colab import userdata
            tok = userdata.get("GITHUB_TOKEN")
            if tok:
                url = AOGC_REPO_URL.replace("https://", f"https://{tok}@")
        except Exception:
            pass
        sh("git", "clone", "--depth", "1", url, "/content/aogc")
    aogc_dir = "/content/aogc"
elif os.path.isdir(AOGC_LOCAL_DIR):
    aogc_dir = AOGC_LOCAL_DIR
elif os.path.exists(AOGC_LOCAL_DIR + ".zip"):
    import zipfile
    os.makedirs("/content/aogc", exist_ok=True)
    with zipfile.ZipFile(AOGC_LOCAL_DIR + ".zip") as z:
        z.extractall("/content/aogc")
    aogc_dir = "/content/aogc"
else:
    raise SystemExit("No token? Either upload AOGC-agent-uq.zip to Drive at "
                     f"{AOGC_LOCAL_DIR}.zip, or add a GITHUB_TOKEN secret, or make the repo public.")
sys.path.insert(0, aogc_dir)
print("aogc repo:", aogc_dir, "| MIRAGE_ROOT:", os.environ["MIRAGE_ROOT"])

In [ ]:
# 6. load the grounding subset
from collections import Counter
from aogc_uq.data import mirage

trajs = mirage.load_dataset(os.environ["MIRAGE_ROOT"],
                            settings=SETTINGS, environments=ENVIRONMENTS, limit=LIMIT)
print(len(trajs), "snapshots")
print(Counter((t.meta["setting"], t.meta["environment"]) for t in trajs))

In [ ]:
# 7. generator (local 4-bit model; downloads once, cached to Drive)
from aogc_uq.rollout import HFGenerator
gen = HFGenerator(MODEL, load_in_4bit=True, max_new_tokens=512, temperature=0.0)

## Judge
Wires the LLM-judge to your API key (stored in Colab Secrets). `stub` runs
label-free (mechanism check only).

In [ ]:
# 8. judge backend
from aogc_uq.rollout import LLMJudge, StubJudge
from google.colab import userdata

if JUDGE_BACKEND == "local":
    # reuse the rollout model as its own judge — free, no API key.
    # NOTE: weaker + not independent; fine for a first pass / pipeline check.
    # For paper-quality labels use a stronger independent judge (API) + human spot-checks.
    judge = LLMJudge(gen.generate)
elif JUDGE_BACKEND == "anthropic":
    from anthropic import Anthropic
    _client = Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
    def _complete(messages):
        system = next((m["content"] for m in messages if m["role"] == "system"), "")
        um = [{"role": m["role"], "content": m["content"]} for m in messages if m["role"] != "system"]
        r = _client.messages.create(model=JUDGE_MODEL, max_tokens=300, system=system, messages=um)
        return r.content[0].text
    judge = LLMJudge(_complete)
elif JUDGE_BACKEND == "openai":
    from openai import OpenAI
    _client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
    def _complete(messages):
        r = _client.chat.completions.create(model=JUDGE_MODEL, messages=messages, max_tokens=300)
        return r.choices[0].message.content
    judge = LLMJudge(_complete)
else:
    judge = StubJudge()
print("judge:", JUDGE_BACKEND)

In [ ]:
# 9. run rollout (resumable: re-run after a disconnect to continue)
import time
from aogc_uq.rollout import run_rollout
out = os.path.join(OUT_DIR, f"scored_{MODEL.split('/')[-1]}.jsonl")
t0 = time.time()
def prog(n, tr):
    if n % 5 == 0:
        print(f"  {n} done  {time.time()-t0:.0f}s  last={tr.task_id[:50]}")
run_rollout(trajs, gen, judge, out_path=out, model_name=MODEL, on_progress=prog)
print("saved ->", out)

## Quick analysis (optional — also runnable on your Mac)
Computes AOGC vs FRANQ-as-agent on the scored set using the **real** DeBERTa-v3
NLI (sentencepiece works on Colab). This is the §3.5b comparison on real data.

Note: the verbalized-confidence baseline needs a confidence-eliciting prompt
variant (TODO) — not included here, so this cell compares AOGC vs FRANQ only.

In [ ]:
# 10. AOGC vs FRANQ on the scored data
import numpy as np
from aogc_uq.rollout import load_scored
from aogc_uq.aogc import get_nli, aogc_step_score
from aogc_uq.baselines import franq_as_agent_uncertainty
from aogc_uq.metrics import auroc, blindspot_auroc
from aogc_uq.data.schema import ErrorType

scored = load_scored(out)
nli = get_nli("MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli")  # real NLI on GPU
aogc, franq, iserr, et = [], [], [], []
for tr in scored:
    for s in tr.steps:
        if s.is_error is None:
            continue
        aogc.append(aogc_step_score(s, nli=nli)["score"])
        franq.append(franq_as_agent_uncertainty(s, nli=nli))
        iserr.append(bool(s.is_error)); et.append(s.error_type)

print("labeled steps:", len(iserr), "| judged errors:", sum(iserr))
if sum(iserr) and sum(1 for x in iserr if not x):
    print("AOGC  blindspot(grounding) AUROC:", round(blindspot_auroc(aogc, iserr, et, ErrorType.GROUNDING), 3))
    print("FRANQ blindspot(grounding) AUROC:", round(blindspot_auroc(franq, iserr, et, ErrorType.GROUNDING), 3))
    print("AOGC  all-error AUROC:", round(auroc(aogc, [int(x) for x in iserr]), 3))
    print("FRANQ all-error AUROC:", round(auroc(franq, [int(x) for x in iserr]), 3))
else:
    print("Need both error and non-error labeled steps for AUROC; raise LIMIT.")

## Next
- Pull `scored_*.jsonl` from Drive to your Mac; rerun analysis there with the
  full `aogc_uq` (metrics, fusion, slicing) — no GPU needed.
- Raise `LIMIT`, add more `SETTINGS`/`ENVIRONMENTS`.
- TODO for full H1: a confidence-eliciting generation variant so the verbalized
  baseline can be compared (the blind-spot headline). Self-consistency / semantic
  entropy baselines via `gen.generate_n(messages, k)`.